# Week 3, Day 2: Cross-Validation

## Motivation
In Day 1, val loss varied significantly depending on which molecules ended up in the validation set.
A single random split gives one performance estimate — but it may be optimistic, pessimistic,
or simply unrepresentative of how the model generalises.

**K-fold cross-validation** addresses this by rotating which portion of the data is used for validation:
- The combined train+val set is split into k equal parts (folds)
- In each iteration, k-1 folds are used for training and 1 fold is used for validation
- This is repeated k times so every fold has served as the validation set exactly once
- The k val loss values are averaged to give a single, more robust performance estimate

The test set remains completely untouched throughout — cross-validation replaces the
validation set, not the test set.

---

## Key Implementation Detail: Normalisation Inside the Fold Loop
In each fold, the training set is different. Normalisation statistics (mean and std) must
be recomputed from the training fold each time, then applied to the validation fold.

Computing normalisation outside the loop using the full train+val set would leak information
from the validation fold into the statistics — the same leakage we were careful to avoid in B1 and B2.

```
For each fold:
    1. Split into train fold and val fold
    2. Compute mean and std from train fold only
    3. Normalise both folds using train fold statistics
    4. Train model, evaluate on val fold
    5. Record val loss
Average val loss across all folds
```

---

## Compute Cost
Cross-validation trains k models instead of 1. For this dataset (1,128 molecules) this is
negligible. For large datasets (100k+) it becomes a real constraint and alternatives like
repeated holdout or nested cross-validation are used instead.

In [3]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

try:
    from rdkit import Chem
except ImportError:
    !pip install rdkit
    from rdkit import Chem

from rdkit.Chem.Descriptors import CalcMolDescriptors
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from sklearn.model_selection import train_test_split, KFold

In [4]:
# ── Load ESOL dataset ────────────────────────────────────────────────────────
url = "https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv"
df = pd.read_csv(url)
compounds = df[["Compound ID", "smiles", "measured log solubility in mols per litre"]].copy()

# ── SMILES → RDKit Mol ───────────────────────────────────────────────────────
def smiles_to_rdkit_mol(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:  # None is returned silently for invalid SMILES
        return "Error"
    return mol

compounds["rdkit_mol"] = compounds["smiles"].apply(smiles_to_rdkit_mol)
error_compounds = compounds[compounds["rdkit_mol"] == "Error"]
print(f"Number of compounds with invalid SMILES: {len(error_compounds)}")
valid_compounds = compounds[compounds["rdkit_mol"] != "Error"]

Number of compounds with invalid SMILES: 0


In [5]:
# ── Compute RDKit descriptors ─────────────────────────────────────────────────
# missingVal=np.nan ensures missing descriptors are flagged rather than silently filled
desc_list = []
for mol in valid_compounds["rdkit_mol"]:
    desc = CalcMolDescriptors(mol, missingVal=np.nan)
    desc_list.append(desc)

desc_df = pd.DataFrame(desc_list)
print(f"Shape before dropping NaN columns: {desc_df.shape}")
print(f"Columns with NaN: {desc_df.isna().any().sum()}")

# Drop columns with any NaN values
desc_df = desc_df.dropna(axis=1)
print(f"Shape after dropping NaN columns: {desc_df.shape}")

Shape before dropping NaN columns: (1128, 217)
Columns with NaN: 0
Shape after dropping NaN columns: (1128, 217)


In [6]:
# ── Build X and y ─────────────────────────────────────────────────────────────
# Extract targets BEFORE dropping from X to prevent data leakage
# ExactMolWt dropped from X as it is highly correlated with MolWt (leakage risk)
y = desc_df[["MolWt", "TPSA", "MolLogP"]].values
X = desc_df.drop(columns=["MolWt", "ExactMolWt", "TPSA", "MolLogP"]).values

print(f"X shape: {X.shape}, y shape: {y.shape}")

X shape: (1128, 213), y shape: (1128, 3)


In [7]:
# ── Hold out test set before cross-validation ─────────────────────────────────
# The test set is carved out once here and never touched during cross-validation.
# Cross-validation operates only on X_trainval / y_trainval.
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42
)

print(f"Train+Val: {X_trainval.shape}, Test: {X_test.shape}")

# ── K-fold cross-validation goes here ────────────────────────────────────────
kf = KFold(n_splits=5, shuffle=True, random_state=42)
features_idx_with_zero_std = set()  # To track features with zero std across all folds
# Pass 1: Collect features with zero std in any fold
for i, (train_index, val_index) in enumerate(kf.split(X_trainval)):
    print(f"Fold {i}:")
    X_train = X_trainval[train_index]
    y_train = y_trainval[train_index]
    X_val = X_trainval[val_index]
    y_val = y_trainval[val_index]
    # print(f"Train length: {len(train_index)}")
    # print(f"Val length: {len(val_index)}")
    # print(f"Train shape: {X_train.shape}")
    # print(f"Val shape: {y_val.shape}")
    # Normalize dataset using training set statistics
    X_train_std = X_train.std(axis=0)
    # Check for zero std to avoid division by zero
    columns_with_zero_std = np.where(X_train_std == 0)[0]
    print(columns_with_zero_std)
    print(f"Number of features with zero std: {len(columns_with_zero_std)}")
    features_idx_with_zero_std.update(columns_with_zero_std)
    # Check for zero std in target set
    y_train_std = y_train.std(axis=0)
    targets_with_zero_std = np.where(y_train_std == 0)[0]
    print("Number of target features with zero std: ", len(targets_with_zero_std))

print(f"Total unique features with zero std across all folds: {len(features_idx_with_zero_std)}")
print(f"Indices of features with zero std: {sorted(features_idx_with_zero_std)}")

Train+Val: (1015, 213), Test: (113, 213)
Fold 0:
[  8  67  80 128 132 136 137 141 152 156 159 165 174 177 178 181 184 189
 202 207 209]
Number of features with zero std: 21
Number of target features with zero std:  0
Fold 1:
[  8  67  80 128 132 136 137 156 159 160 165 166 174 177 178 181 184 189
 202 207 209]
Number of features with zero std: 21
Number of target features with zero std:  0
Fold 2:
[  8  67  80 128 132 136 137 156 159 165 174 177 178 181 184 189 202 207
 209]
Number of features with zero std: 19
Number of target features with zero std:  0
Fold 3:
[  8  67  80 128 132 136 137 156 159 165 174 177 178 181 184 189 202 207
 209 210]
Number of features with zero std: 20
Number of target features with zero std:  0
Fold 4:
[  8  67  80 128 132 136 137 156 159 165 174 177 178 181 184 189 198 202
 207 209]
Number of features with zero std: 20
Number of target features with zero std:  0
Total unique features with zero std across all folds: 25
Indices of features with zero std: [np

In [8]:
# DataLoaders
class MoleculeDataset(Dataset):
    def __init__(self, descriptors, labels):
        self.descriptors = descriptors
        self.labels = labels
    
    def __len__(self):
        return len(self.descriptors)
    
    def __getitem__(self, idx):
        return self.descriptors[idx], self.labels[idx]


In [9]:
def build_model(input_dim: int):
    '''
    Function for initializing model for each fold    
    '''
    return nn.Sequential(
    # --- Block 1 ---
    # Projects 194 raw input features (data, not neurons) into 512 neurons
    nn.Linear(input_dim, 512),
    nn.BatchNorm1d(512),  # normalises the 512 neuron outputs
    nn.ReLU(),            # introduces non-linearity
    nn.Dropout(0.2),      # randomly deactivates 20% of neurons during training

    # --- Block 2 ---
    # Hidden layer: 512 neurons → 256 neurons
    nn.Linear(512, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(0.2),

    # --- Block 3 ---
    # Hidden layer: 256 neurons → 128 neurons
    nn.Linear(256, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(0.2),

    # --- Output ---
    # No activation, BatchNorm, or Dropout — raw value for regression
    nn.Linear(128, 3)
)


In [ ]:
# Pass 2: Training loop
torch.manual_seed(42)
# Create a mask for dropping features with zero std
zero_std_mask = np.zeros(X_trainval.shape[1], dtype=bool)
zero_std_mask[sorted(features_idx_with_zero_std)] = True
fold_val_losses = []
for i, (train_index, val_index) in enumerate(kf.split(X_trainval)):
    print(f"Fold {i}:")
    # Drop features with zero std
    X_train_clean = X_trainval[train_index][:, ~zero_std_mask]
    y_train = y_trainval[train_index]
    X_val_clean = X_trainval[val_index][:, ~zero_std_mask]
    y_val = y_trainval[val_index]
    # Normalize dataset using training set statistics
    X_train_clean_mean = X_train_clean.mean(axis=0)
    X_train_clean_std = X_train_clean.std(axis=0)
    ##
    X_train_norm = (X_train_clean - X_train_clean_mean) / X_train_clean_std
    X_val_norm = (X_val_clean - X_train_clean_mean) / X_train_clean_std
    # Normalize target dataset
    y_train_mean = y_train.mean(axis=0)
    y_train_std = y_train.std(axis=0)
    ## 
    y_train_norm = (y_train - y_train_mean) / y_train_std
    y_val_norm = (y_val - y_train_mean) / y_train_std

    # Convert into tensors
    X_train_tensor = torch.tensor(X_train_norm, dtype=torch.float32)
    X_val_tensor = torch.tensor(X_val_norm, dtype=torch.float32)

    y_train_tensor = torch.tensor(y_train_norm, dtype=torch.float32)
    y_val_tensor = torch.tensor(y_val_norm, dtype=torch.float32)

    print(f"X_Train: {X_train_tensor.shape}, y_train: {y_train_tensor.shape}")

    # Create Dataset & DataLoader
    X_train_dataset = MoleculeDataset(X_train_tensor, y_train_tensor)
    X_val_dataset = MoleculeDataset(X_val_tensor, y_val_tensor)

    X_train_loader = DataLoader(X_train_dataset, batch_size=32, shuffle=True)
    X_val_loader = DataLoader(X_val_dataset, batch_size=32, shuffle=False)

    # model, loss function and optimizer
    full_model = build_model(X_train_tensor.shape[1])
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(full_model.parameters(), lr=0.001)
    # Learning rate decay + early stopping setup
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,      # halve the learning rate
        patience=5,      # after 5 epochs with no improvement
        min_lr=1e-6      # never go below this
    )

    num_epochs = 100
    best_val_loss = float('inf')
    early_stopping_patience = 10
    patience_counter = 0

    for epoch in range(num_epochs):
        # --- Training loop ---
        full_model.train() # Set model to training mode (enables dropout, batchnorm updates)
        train_loss = 0.0
        for batch in X_train_loader:
            X_batch, y_batch = batch
            optimizer.zero_grad()           # Clear gradients
            outputs = full_model(X_batch)   # Forward pass
            loss = criterion(outputs, y_batch) # Compute loss
            loss.backward()                 # Backpropagation
            optimizer.step()                # Update weights
            train_loss += loss.item() # Accumulate loss
        
        # --- Validation loop ---
        full_model.eval() # Set model to evaluation mode (disables dropout, batchnorm updates
        val_loss = 0.0
        with torch.no_grad(): # No need to compute gradients during validation
            for batch in X_val_loader:
                X_batch, y_batch = batch
                outputs = full_model(X_batch)   # Forward pass
                loss = criterion(outputs, y_batch) # Compute loss
                val_loss += loss.item() # Accumulate loss
        
        # --- Early stopping check ---
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(full_model.state_dict(), f'fold_{i}_best_model.pt')  # ← save here
            print(f"Epoch {epoch}/{num_epochs}, Train Loss: {train_loss/len(X_train_loader):.4f}, Val Loss: {val_loss/len(X_val_loader):.4f}")
        else:
            # weights here do not reflect that of the best epoch
            # That's why we save the weights at the best epoch, not here
            patience_counter += 1
            if patience_counter >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch}. Best Val Loss: {val_loss/len(X_val_loader):.4f}")
                break  # weights already saved at best epoch

        scheduler.step(val_loss / len(X_val_loader))  # ← after the if/else, once per epoch
    # Restore best weights after training
    full_model.load_state_dict(torch.load(f'fold_{i}_best_model.pt'))
    fold_val_losses.append(best_val_loss / len(X_val_loader))

print(F"Mean val losses across folds: {np.mean(fold_val_losses):.4f} +- {np.std(fold_val_losses):.4f}")



    


Fold 0:
X_Train: torch.Size([812, 188]), y_train: torch.Size([812, 3])
Epoch 0/100, Train Loss: 0.3138, Val Loss: 0.0773
Epoch 1/100, Train Loss: 0.1575, Val Loss: 0.0514
Epoch 3/100, Train Loss: 0.1312, Val Loss: 0.0466
Epoch 4/100, Train Loss: 0.1265, Val Loss: 0.0346
Epoch 10/100, Train Loss: 0.0923, Val Loss: 0.0340
Epoch 11/100, Train Loss: 0.0907, Val Loss: 0.0313
Epoch 13/100, Train Loss: 0.0972, Val Loss: 0.0297
Epoch 21/100, Train Loss: 0.0925, Val Loss: 0.0283
Epoch 23/100, Train Loss: 0.0749, Val Loss: 0.0262
Epoch 24/100, Train Loss: 0.0727, Val Loss: 0.0255
Epoch 29/100, Train Loss: 0.0743, Val Loss: 0.0225
Early stopping at epoch 39. Best Val Loss: 0.0232
Fold 1:
X_Train: torch.Size([812, 188]), y_train: torch.Size([812, 3])
Epoch 0/100, Train Loss: 0.3046, Val Loss: 0.0601


/tmp/ipykernel_69322/1689685407.py:103: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  fold_rmse.append(torch.sqrt(torch.tensor(fold_val_losses[-1]) * y_train_std + y_train_mean))


Epoch 1/100, Train Loss: 0.1537, Val Loss: 0.0483
Epoch 2/100, Train Loss: 0.1379, Val Loss: 0.0455
Epoch 5/100, Train Loss: 0.1161, Val Loss: 0.0344
Epoch 6/100, Train Loss: 0.1036, Val Loss: 0.0342
Epoch 7/100, Train Loss: 0.1097, Val Loss: 0.0304
Epoch 8/100, Train Loss: 0.0975, Val Loss: 0.0225
Epoch 12/100, Train Loss: 0.0958, Val Loss: 0.0192
Epoch 22/100, Train Loss: 0.0773, Val Loss: 0.0182
Epoch 29/100, Train Loss: 0.0749, Val Loss: 0.0155
Early stopping at epoch 39. Best Val Loss: 0.0180
Fold 2:
X_Train: torch.Size([812, 188]), y_train: torch.Size([812, 3])
Epoch 0/100, Train Loss: 0.3130, Val Loss: 0.1051
Epoch 1/100, Train Loss: 0.1686, Val Loss: 0.0469
Epoch 5/100, Train Loss: 0.1052, Val Loss: 0.0459
Epoch 9/100, Train Loss: 0.1015, Val Loss: 0.0413
Epoch 10/100, Train Loss: 0.0840, Val Loss: 0.0362
Epoch 11/100, Train Loss: 0.1029, Val Loss: 0.0356
Epoch 12/100, Train Loss: 0.0991, Val Loss: 0.0286
Epoch 13/100, Train Loss: 0.0955, Val Loss: 0.0275
Epoch 18/100, Train Lo